<a href="https://colab.research.google.com/github/cellatlas/human/blob/master/markers/adipose/markers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gget
!pip install -q git+https://github.com/sbooeshaghi/ec



[notice] A new release of pip is available: 23.3.2 -> 24.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.3.2 -> 24.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from ec.utils import write_markers

In [4]:
!head genes.txt

In [11]:
genes = pd.read_csv('../../reference/genes.txt',sep="\t",header = None, names=["gid", "gname"]).dropna()

# Adipose

In [20]:
import json
with open("citation.json", 'r') as f:
    d = json.load(f)

In [21]:
d

{'doi': 'https://doi.org/10.1038/s42255-019-0152-6',
 'citation': 'Vijay, J., Gauthier, MF., Biswell, R.L. et al. Single-cell analysis of human adipose tissue identifies depot- and disease-specific cell types. Nat Metab 2, 97–109 (2020).',
 'tables': ['https://static-content.springer.com/esm/art%3A10.1038%2Fs42255-019-0152-6/MediaObjects/42255_2019_152_MOESM3_ESM.xlsx'],
 'organ': 'adipose',
 'species': 'homo_sapiens',
 'reference': 'GRCh38'}

In [24]:
# don't include in header
table_name = "Supplementary Table 2"

header = [
    {
      "species": d["species"],
      "organ": d["organ"],
      "reference": d["reference"],
      "paper_doi": d["doi"],
      "table_link": ",".join(d["tables"])
    }
]

In [25]:
# mx diff cols
# group
# "target",
# "rank",
# "nnz",
# "nnz_frac",
# "mean",
# "log_fc",
# "p_raw",
# "p_corr",

rename = {
    "P value": "p_raw",
    "Average LogFC": "log_fc",
    "Percentage of cells in cluster expressing the gene": "nnz_frac",
    "Adj. P value": "p_corr",
    "Cluster#": "celltype_num",
    "Gene": "target",
    "Cluster name": "group"
}

In [87]:
df = pd.read_excel(d['tables'][0], sheet_name=table_name, skiprows=2).rename(columns=rename)
name_map = pd.read_excel(d['tables'][0], sheet_name="Supplementary Table 4", skiprows=2).rename(columns={"Cluster": "group", "Manual Annotation": "group_name", "Unsupervised Annotation1 (a)": "group_name_detailed"}).dropna()
name_map = name_map[["group", "group_name"]].drop_duplicates().set_index("group")
df['group'] = df.group.map(name_map["group_name"])

In [88]:
bidx = df['target'].isin(genes['gname'])
print(f'Filtered {np.sum(~bidx)} out of {len(bidx)} genes')
df = df[bidx]

Filtered 42 out of 1320 genes


In [89]:
df.head()

,p_raw,log_fc,nnz_frac,Percentage of cells in all other clusters expressing the gene,p_corr,celltype_num,target,group
0,0.0,2.365577,0.824,0.088,0.0,0,KRT18,Progenitor/ Stem cells
1,0.0,2.050661,0.674,0.059,0.0,0,KRT8,Progenitor/ Stem cells
2,0.0,1.911774,0.833,0.111,0.0,0,ITLN1,Progenitor/ Stem cells
3,0.0,1.718079,0.774,0.123,0.0,0,TM4SF1,Progenitor/ Stem cells
4,0.0,1.713866,0.542,0.052,0.0,0,KRT19,Progenitor/ Stem cells


In [94]:
min_mean = 100
max_pval = 1e-5
min_lfc = 2
max_gene_shares = 5
max_per_celltype = 20

# filter by criteria
dfc = df.query(f"p_raw <= {max_pval} & log_fc >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell type
non_repeat_genes = dfc["target"].value_counts()[dfc["target"].value_counts() < max_gene_shares].index.values

m = dfc[dfc.target.isin(non_repeat_genes)].sort_values('nnz_frac', ascending = True)

# max number to sample is equal to the max number of genes across all celltype
n_sample = max(m["group"].value_counts().min(), max_per_celltype)

# sample n_sample genes
markers = m.groupby('group').tail(n_sample)
markers_dict = markers.groupby("group")["target"].apply(lambda x: list(x)).to_dict()

In [95]:
markers.group.value_counts()

Immune cells              20
Progenitor/ Stem cells    12
Endothelial cells         12
Name: group, dtype: int64

In [96]:
write_markers("markers.txt", markers_dict, header)

In [98]:
markers.groupby("group")["nnz_frac"].mean().sort_values()

group
Endothelial cells         0.581417
Progenitor/ Stem cells    0.626333
Immune cells              0.831150
Name: nnz_frac, dtype: float64

In [99]:
!cat markers.txt

# homo_sapiens	adipose	GRCh38	https://doi.org/10.1038/s42255-019-0152-6	https://static-content.springer.com/esm/art%3A10.1038%2Fs42255-019-0152-6/MediaObjects/42255_2019_152_MOESM3_ESM.xlsx
Endothelial cells	APOLD1,SELE,IL6,CLDN5,VWF,AKAP12,TFPI,RBP7,ACKR1,CCL21,FABP4,TFF3
Immune cells	CXCL8,S100A9,SPP1,CXCL3,LIPA,CXCL8,CCL5,NKG7,CTSB,CTSD,HLA-DRB1,HLA-DPA1,CSTB,HLA-DPB1,FABP5,TYROBP,HLA-DRA,FABP4,CD74,FTL
Progenitor/ Stem cells	PTN,PI16,TPPP3,PTGDS,FBN1,DPT,KRT8,MFAP5,KRT18,CXCL14,IGFBP6,CFD
